# Day 3 — Explicit p=1 Penalty-X circuit

This notebook makes one circuit layer and its state evolution transparent. It performs **no parameter optimization** and uses only the predetermined diagnostic pair.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from circuit import build_p1_qaoa_circuit, primitive_gate_counts, statevector_at_checkpoints
from day3_artifacts import DIAGNOSTIC_BETA, DIAGNOSTIC_GAMMA, DIAGNOSTIC_LABEL
from exact_reference import networkx_shortest_reference
from graph import load_graph, path_to_edge_bitstring
from ising import qubo_to_ising
from qubo import build_qubo, edge_vector_to_state_index, enumerate_state_space
from statevector_reference import (compare_statevectors, probabilities,
    reference_statevector_at_checkpoints, total_variation_distance)

## 1. Load the frozen contracts

Day 3 consumes the graph, penalty grid, and Ising model without modifying them.

In [ ]:
graph = load_graph(PROJECT_ROOT / 'data' / 'graph.json')
penalty_contract = json.loads((PROJECT_ROOT / 'data' / 'penalty_contract.json').read_text())
circuit_contract = json.loads((PROJECT_ROOT / 'data' / 'circuit_contract.json').read_text())
exact = networkx_shortest_reference(graph)
assert penalty_contract['A_crit'] == 5
assert [record['A'] for record in penalty_contract['penalty_values']] == [2, 5, 6, 12]
{'exact route': exact.node_path, 'C*': exact.cost,
 'qubits': circuit_contract['qubit_count'],
 'penalty grid': [record['A'] for record in penalty_contract['penalty_values']]}

## 2. Factor-of-two gate convention

Qiskit defines `RZ(θ)=exp(−iθZ/2)` and `RZZ(θ)=exp(−iθZZ/2)`. Therefore `h_i Z_i` becomes `RZ(2γ₁h_i)` and `J_ij Z_iZ_j` becomes `RZZ(2γ₁J_ij)`. The mixer uses `RX(2β₁)` on every qubit. The `c₀I` term contributes only a global phase and has no physical gate.

## 3. Construct the explicit parameterized circuit

A=6 is used for mechanism visualization because it is the frozen just-supercritical classical case—not because of QAOA performance.

In [ ]:
A = 6
hamiltonian = qubo_to_ising(build_qubo(graph, A))
circuit = build_p1_qaoa_circuit(graph, hamiltonian)
print('free parameters:', sorted(parameter.name for parameter in circuit.parameters))
print('primitive gates:', primitive_gate_counts(circuit))
print('layers:', circuit.metadata['cost_layer_count'], 'cost +',
      circuit.metadata['mixer_layer_count'], 'mixer')

## 4. Predetermined diagnostic parameters

The same `γ=π/7`, `β=π/11` pair is used for every frozen A. It was not searched or selected by performance.

In [ ]:
print(DIAGNOSTIC_LABEL, 'gamma=', DIAGNOSTIC_GAMMA, 'beta=', DIAGNOSTIC_BETA)
states = enumerate_state_space(graph)
qiskit_states = statevector_at_checkpoints(
    graph, hamiltonian, gamma=DIAGNOSTIC_GAMMA, beta=DIAGNOSTIC_BETA)
numpy_states = reference_statevector_at_checkpoints(
    states, penalty=A, gamma=DIAGNOSTIC_GAMMA, beta=DIAGNOSTIC_BETA)
rows = []
for checkpoint in ('initial', 'post_cost', 'post_mixer'):
    comparison = compare_statevectors(numpy_states[checkpoint], qiskit_states[checkpoint])
    rows.append({'checkpoint': checkpoint, 'fidelity': comparison.fidelity,
                 'max amplitude error': comparison.max_amplitude_absolute_error,
                 'NumPy norm': np.linalg.norm(numpy_states[checkpoint]),
                 'Qiskit norm': np.linalg.norm(qiskit_states[checkpoint])})
display(pd.DataFrame(rows))

## 5. Cost phase, then mixer interference

The cost layer is diagonal: it changes phases while leaving computational-basis probabilities unchanged. The X mixer then combines amplitudes and redistributes probability.

In [ ]:
p_initial = probabilities(numpy_states['initial'])
p_cost = probabilities(numpy_states['post_cost'])
p_mixer = probabilities(numpy_states['post_mixer'])
relative_phases = np.angle(numpy_states['post_cost'] / numpy_states['initial'])
print('max probability change after cost =', np.max(np.abs(p_cost - p_initial)))
print('distinct cost phases (rounded) =', len(np.unique(np.round(relative_phases, 12))))
print('post-cost to post-mixer total variation =', total_variation_distance(p_cost, p_mixer))
print(DIAGNOSTIC_LABEL, '— mechanism visualization only; no performance claim')

## 6. Presentation figures

In [ ]:
display(Image(filename=PROJECT_ROOT / 'figures' / '05_explicit_p1_qaoa_circuit.png', width=1000))
display(Image(filename=PROJECT_ROOT / 'figures' / '06_p1_quantum_state_evolution.png', width=1000))